# INSTRUCTOR SOLUTIONS — DO NOT DISTRIBUTE

## Task 07b: Building Regression Models
## Predicting Car Fuel Economy

This is the complete solution with all code filled in.

## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error

print("Libraries loaded!")

In [ ]:
cars = pd.read_excel('MY26 FE Guide for DOE.xlsx', sheet_name='FEguide')
cars = cars[['Mfr Name', 'Division', 'Carline', 'Eng Displ', '# Cyl',
             'Drive Sys', 'Comb FE (Guide) - Conventional Fuel']].copy()
cars.columns = ['manufacturer', 'brand', 'model',
                'engine_liters', 'cylinders',
                'drive', 'mpg']
cars = cars.dropna()
cars = cars[cars['mpg'] > 0]

print(f"Loaded {len(cars)} vehicles")
print(f"\nFirst 5 rows:")
print(cars.head())

## Part 1: Explore the Data

In [ ]:
# Basic statistics
print("Descriptive Statistics:")
print(cars[['engine_liters', 'cylinders', 'mpg']].describe().round(1))

In [ ]:
# Drive types
print("\nDrive types in data:")
print(cars['drive'].value_counts())
print(f"\nUnique drives: {sorted(cars['drive'].unique())}")

## Part 2: Correlation Analysis

In [ ]:
correlation = cars[['engine_liters', 'cylinders', 'mpg']].corr()
print("Correlation Matrix:")
print(correlation.round(3))

In [ ]:
# Heatmap
plt.figure(figsize=(6, 5))
sns.heatmap(correlation,
            annot=True,
            cmap='coolwarm',
            center=0,
            fmt='.2f',
            vmin=-1, vmax=1)
plt.title('Correlation: Car Features & MPG', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(cars['engine_liters'],
            cars['mpg'],
            alpha=0.4, color='steelblue', s=40)
plt.xlabel('Engine Size (Liters)', fontsize=11)
plt.ylabel('Combined MPG', fontsize=11)
plt.title('Bigger Engines = Worse Fuel Economy', fontsize=12, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

r_engine_mpg = cars['engine_liters'].corr(cars['mpg'])
print(f"Pearson r between engine_liters and mpg: {r_engine_mpg:.4f}")
print(f"Interpretation: Moderate negative correlation")

## Part 3: Feature Engineering

In [ ]:
# One-hot encode drive type
cars_encoded = pd.get_dummies(cars, columns=['drive'], drop_first=True)

print(f"New columns after encoding:")
print([c for c in cars_encoded.columns if c.startswith('drive_')])
print(f"\nTotal columns now: {len(cars_encoded.columns)}")

In [ ]:
# SOLUTION: Using Option B (engine_liters + cylinders)
# This is a good balance between simplicity and features
featureColumns = ['engine_liters', 'cylinders']

print(f"Feature columns: {featureColumns}")
print(f"\nNote: Option C (with drive type) would also work.")
print(f"We use Option B for clarity in teaching.")

### Instructor Note
The three options provide different levels of complexity:
- **Option A** (engine_liters only): Very simple, R² ≈ 0.60
- **Option B** (engine_liters + cylinders): Good balance, R² ≈ 0.68
- **Option C** (all features): More complex, R² ≈ 0.73

All are reasonable. Option B is used here because cylinders adds slightly more info than drive type, and students can understand it more easily.

## Part 4: Build the Model

In [ ]:
X = cars_encoded[featureColumns]
y = cars_encoded['mpg']

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training set: {len(X_train)} vehicles")
print(f"Test set: {len(X_test)} vehicles")

In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)

print("✓ Model trained!")

## Part 5: Evaluate the Model

In [ ]:
predictions = model.predict(X_test)
r2 = r2_score(y_test, predictions)
mae = mean_absolute_error(y_test, predictions)

print(f"R² Score:  {r2:.4f}  ({r2*100:.1f}%)")
print(f"MAE:       {mae:.2f} MPG")

print(f"\nInterpretation:")
print(f"  Our model explains {r2*100:.1f}% of MPG variation.")
print(f"  Predictions are off by {mae:.2f} MPG on average.")

print(f"\nComparison to Weather Model:")
print(f"  Weather R² = 0.92  (too easy — same-day temperatures)")
print(f"  Cars R² = {r2:.2f}  (realistic — multiple factors)")

In [ ]:
# Predictions vs Actual
plt.figure(figsize=(8, 6))

# Perfect prediction line
minMpg = y_test.min()
maxMpg = y_test.max()
plt.plot([minMpg, maxMpg],
         [minMpg, maxMpg],
         'r--', linewidth=2, label='Perfect predictions')

# Our predictions
plt.scatter(y_test, predictions,
            alpha=0.5, color='steelblue', s=40,
            label='Our predictions')

plt.xlabel('Actual MPG', fontsize=11)
plt.ylabel('Predicted MPG', fontsize=11)
plt.title('How Close Are Our Predictions?', fontsize=12, fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Analysis:")
print(f"  Most points are within {mae:.1f} MPG of the diagonal")
print(f"  A few outliers exist (unusual cars)")
print(f"  No systematic bias (not consistently above or below)")

## Part 6: Coefficients

In [ ]:
coefficientTable = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': model.coef_
})

print("What the model learned:\n")
print(f"Intercept: {model.intercept_:.2f}\n")
for _, row in coefficientTable.iterrows():
    print(f"{row['Feature']:20s}  {row['Coefficient']:+.4f}")

print("\nInterpretation:")
for _, row in coefficientTable.iterrows():
    impact = "decreases" if row['Coefficient'] < 0 else "increases"
    print(f"  - Each +1 unit of {row['Feature']:16s} → {impact} MPG by {abs(row['Coefficient']):.3f}")

In [ ]:
# Bar chart
plt.figure(figsize=(8, 4))
colors = ['red' if c < 0 else 'green' for c in coefficientTable['Coefficient']]
plt.barh(coefficientTable['Feature'],
         coefficientTable['Coefficient'],
         color=colors)
plt.xlabel('Effect on MPG', fontsize=11)
plt.title('Which Features Matter Most?', fontsize=12, fontweight='bold')
plt.axvline(x=0, color='black', linewidth=0.8)
plt.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

print("Green = increases MPG (good)")
print("Red = decreases MPG (bad)")

## Part 7: Sample Student Answers

In [ ]:
print("SAMPLE STUDENT RESPONSES:\n")
print(f"1. R² value: {r2:.3f}")
print(f"   This means the model explains {r2*100:.1f}% of MPG variation.\n")

print(f"2. Comparison to weather (R² = 0.92 vs cars R² = {r2:.2f}):")
print(f"   - The cars model is LOWER (as expected)")
print(f"   - Why: Weather model predicted same-day temps (temp_min & temp_max)") 
print(f"     which are nearly identical measurements.")
print(f"   - Cars model predicts based on engine design specs, which are less")
print(f"     deterministic — many other factors (weight, aerodynamics, tech) matter.\n")

max_coef_idx = abs(coefficientTable['Coefficient']).idxmax()
max_feature = coefficientTable.loc[max_coef_idx, 'Feature']
max_coef = coefficientTable.loc[max_coef_idx, 'Coefficient']
print(f"3. Biggest impact: {max_feature}")
print(f"   - Sign: NEGATIVE (bigger engines = worse MPG)")
print(f"   - Each +1 liter of engine size = {max_coef:.3f} MPG penalty\n")

print(f"4. MAE: {mae:.2f} MPG")
print(f"   - On average, predictions are off by about {mae:.0f} MPG")
print(f"   - For a car that gets 25 MPG, prediction might be 20-30 MPG\n")

print(f"5. Is it good enough for real world?")
print(f"   - Yes, for general purposes. R² = {r2:.2f} explains 2/3 of variation.")
print(f"   - No, for precise predictions. Error is ±{mae:.1f} MPG is significant.")
print(f"   - Depends on use case: car shopping vs engineering specs.")

## Part 8: Making Predictions

In [ ]:
# Example predictions
print("EXAMPLE PREDICTIONS:\n")

car_examples = [
    {'engine_liters': [2.0], 'cylinders': [4], 'description': '2.0L 4-cylinder (typical sedan)'},
    {'engine_liters': [3.5], 'cylinders': [6], 'description': '3.5L 6-cylinder (midsize SUV)'},
    {'engine_liters': [5.0], 'cylinders': [8], 'description': '5.0L 8-cylinder (pickup truck)'},
]

for car_dict in car_examples:
    desc = car_dict.pop('description')
    test_car = pd.DataFrame(car_dict)
    pred = model.predict(test_car)[0]
    print(f"{desc}")
    print(f"  Predicted MPG: {pred:.1f}")
    print()

## Part 9: Challenge — Comparing Feature Sets

In [ ]:
# Compare R² across different feature combinations
print("COMPARING FEATURE SETS:\n")

feature_sets = [
    (['engine_liters'], 'Option A: Engine size only'),
    (['cylinders'], 'Option A2: Cylinders only'),
    (['engine_liters', 'cylinders'], 'Option B: Engine + Cylinders'),
]

results = []
for features, label in feature_sets:
    X_test_subset = X_test[features]
    preds_subset = model.predict(X_test_subset)
    r2_subset = r2_score(y_test, preds_subset)
    mae_subset = mean_absolute_error(y_test, preds_subset)
    results.append((label, r2_subset, mae_subset))
    print(f"{label}")
    print(f"  R²: {r2_subset:.4f}  MAE: {mae_subset:.2f}")
    print()

print("Key insight: Adding cylinders to engine_liters improves R²")
print("because they capture different aspects of engine power.")
print("But the improvement diminishes as we add more features.")

## Grading Rubric

**Part 1-2: Exploration (10 points)**
- Correct descriptive statistics
- Correct drive type analysis

**Part 3: Correlation (10 points)**
- Correct heatmap creation
- Correct Pearson r calculation

**Part 5: Model Evaluation (15 points)**
- Correct predictions
- Correct R² and MAE
- Correct interpretation of results

**Part 6: Coefficients (10 points)**
- Correct bar chart
- Correct interpretation of each coefficient

**Part 7: Final Interpretation (15 points)**
- Thoughtful comparison to weather model
- Reasonable answer about real-world applicability
- Understanding that lower R² doesn't mean bad model

**Part 8: Prediction (10 points)**
- Correct prediction values
- Reasonable interpretation

**Part 9: Challenge (10 points)**
- Any valid attempt to compare features
- Reasonable conclusions

**Total: 80 points**